In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.baselines import SoftAttentionUNet, StaticUNet
from src.dataloaders import get_dataloaders
from src.dynamic_unet import DynamicUNet
from src.metrics import compute_segmentation_metrics
from src.transforms import TRAIN_MEAN, TRAIN_STD

RANDOM_SEED = 67
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = "cuda"
print(f"Evaluation device: {device}")

In [ ]:
static_model = StaticUNet().to(device)
attention_model = SoftAttentionUNet().to(device)
dynamic_model = DynamicUNet().to(device)

static_checkpoint = torch.load(
    "models/best_static_unet.pth", map_location=device, weights_only=True
)["model_state_dict"]
attention_checkpoint = torch.load(
    "models/best_attention_unet.pth", map_location=device, weights_only=True
)["model_state_dict"]
dynamic_checkpoint = torch.load(
    "models/best_dynamic_unet.pth", map_location=device, weights_only=True
)["model_state_dict"]
static_model.load_state_dict(static_checkpoint)
attention_model.load_state_dict(attention_checkpoint)
dynamic_model.load_state_dict(dynamic_checkpoint)

static_model.eval()
attention_model.eval()
dynamic_model.eval()
_, _, test_loader = get_dataloaders(batch_size=1, num_workers=4)

In [ ]:
def convolution_gflops(
    spatial_size: int, input_channels: int, output_channels: int, kernel_size: int
) -> float:
    return (
        2.0
        * spatial_size
        * spatial_size
        * input_channels
        * output_channels
        * kernel_size
        * kernel_size
        / 1e9
    )


def decoder_transpose_gflops(
    spatial_size: int, input_channels: int, output_channels: int, kernel_size: int = 2
) -> float:
    input_spatial_size = spatial_size // 2
    return (
        2.0 * input_spatial_size * input_spatial_size * input_channels
        * output_channels * kernel_size * kernel_size / 1e9
    )


decoder_stage_specs = (
    (64, 1024, 512),
    (128, 512, 256),
    (256, 256, 128),
    (512, 128, 64),
)
static_decoder_gflops = 0.0
attention_decoder_gflops = 0.0

for spatial_size, input_channels, output_channels in decoder_stage_specs:
    transposed_convolution_cost = decoder_transpose_gflops(
        spatial_size, input_channels, output_channels
    )
    first_decoder_convolution_cost = convolution_gflops(
        spatial_size, output_channels * 2, output_channels, 3
    )
    second_decoder_convolution_cost = convolution_gflops(
        spatial_size, output_channels, output_channels, 3
    )
    stage_dense_cost = (
        transposed_convolution_cost
        + first_decoder_convolution_cost
        + second_decoder_convolution_cost
    )
    static_decoder_gflops += stage_dense_cost
    attention_decoder_gflops += stage_dense_cost

    intermediate_channels = output_channels // 2
    attention_gate_cost = (
        convolution_gflops(spatial_size, output_channels, intermediate_channels, 1)
        + convolution_gflops(spatial_size, output_channels, intermediate_channels, 1)
        + convolution_gflops(spatial_size, intermediate_channels, 1, 1)
    )
    attention_decoder_gflops += attention_gate_cost

output_head_gflops = convolution_gflops(512, 64, 1, 1)
static_decoder_gflops += output_head_gflops
attention_decoder_gflops += output_head_gflops


def dynamic_decoder_gflops(active_fractions: list[float]) -> float:
    coarse_resolution = 64
    total_cost = decoder_transpose_gflops(coarse_resolution, 1024, 512)
    total_cost += convolution_gflops(coarse_resolution, 1024, 512, 3)
    total_cost += convolution_gflops(coarse_resolution, 512, 512, 3)

    gated_stage_specs = (
        (128, 512, 256),
        (256, 256, 128),
        (512, 128, 64),
    )
    for (spatial_size, input_channels, output_channels), active_fraction in zip(
        gated_stage_specs, active_fractions
    ):
        total_cost += decoder_transpose_gflops(
            spatial_size, input_channels, output_channels
        )
        total_cost += convolution_gflops(spatial_size, output_channels, 1, 1)
        total_cost += convolution_gflops(
            spatial_size, output_channels * 2, output_channels, 1
        )
        heavy_convolution_cost = 2.0 * convolution_gflops(
            spatial_size, output_channels, output_channels, 3
        )
        background_projection_cost = convolution_gflops(
            spatial_size, output_channels, output_channels, 1
        )
        total_cost += active_fraction * heavy_convolution_cost
        total_cost += (1.0 - active_fraction) * background_projection_cost

    total_cost += output_head_gflops
    return total_cost


static_counts = {"true_positives": 0, "false_positives": 0, "false_negatives": 0}
attention_counts = {"true_positives": 0, "false_positives": 0, "false_negatives": 0}

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True) >= 0.5
        static_predictions = torch.sigmoid(static_model(inputs)) >= 0.5
        attention_predictions = torch.sigmoid(attention_model(inputs)) >= 0.5
        for predictions, counts in (
            (static_predictions, static_counts),
            (attention_predictions, attention_counts),
        ):
            counts["true_positives"] += int((predictions & targets).sum().item())
            counts["false_positives"] += int((predictions & ~targets).sum().item())
            counts["false_negatives"] += int((~predictions & targets).sum().item())


def f1_from_counts(counts: dict[str, int]) -> float:
    true_positives = counts["true_positives"]
    false_positives = counts["false_positives"]
    false_negatives = counts["false_negatives"]
    denominator = 2 * true_positives + false_positives + false_negatives
    return 2.0 * true_positives / denominator if denominator else 0.0


static_f1 = f1_from_counts(static_counts)
attention_f1 = f1_from_counts(attention_counts)
print(f"Static U-Net: F1={static_f1:.5f}, decoder GFLOPs={static_decoder_gflops:.3f}")
print(
    f"Attention U-Net: F1={attention_f1:.5f}, "
    f"decoder GFLOPs={attention_decoder_gflops:.3f}"
)

In [ ]:
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
dynamic_f1_scores = []
dynamic_gflops = []
dynamic_active_percentages = []

for tau in thresholds:
    dynamic_model.set_entropy_threshold(tau)
    dynamic_counts = {
        "true_positives": 0,
        "false_positives": 0,
        "false_negatives": 0,
    }
    stage_active_pixels = [0.0, 0.0, 0.0]
    stage_total_pixels = [0, 0, 0]
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True) >= 0.5
            logits, _, routing_masks_list = dynamic_model(
                inputs, return_internals=True
            )
            predictions = torch.sigmoid(logits) >= 0.5
            dynamic_counts["true_positives"] += int((predictions & targets).sum().item())
            dynamic_counts["false_positives"] += int((predictions & ~targets).sum().item())
            dynamic_counts["false_negatives"] += int((~predictions & targets).sum().item())
            for stage_index, routing_mask in enumerate(routing_masks_list):
                stage_active_pixels[stage_index] += float(routing_mask.sum().item())
                stage_total_pixels[stage_index] += routing_mask.numel()

    active_fractions = [
        active_pixels / total_pixels if total_pixels else 0.0
        for active_pixels, total_pixels in zip(stage_active_pixels, stage_total_pixels)
    ]
    dynamic_f1 = f1_from_counts(dynamic_counts)
    average_active_percentage = 100.0 * float(np.mean(active_fractions))
    empirical_decoder_gflops = dynamic_decoder_gflops(active_fractions)
    dynamic_f1_scores.append(dynamic_f1)
    dynamic_gflops.append(empirical_decoder_gflops)
    dynamic_active_percentages.append(average_active_percentage)

print(f"{'Model':<22} {'Threshold':>10} {'F1 Score':>12} {'Decoder GFLOPs':>17} {'Active 3x3':>14}")
print(f"{'Static U-Net':<22} {'—':>10} {static_f1:>12.5f} {static_decoder_gflops:>17.3f} {'100.00%':>14}")
print(f"{'Attention U-Net':<22} {'—':>10} {attention_f1:>12.5f} {attention_decoder_gflops:>17.3f} {'100.00%':>14}")
for tau, f1_score, gflops, active_percentage in zip(
    thresholds, dynamic_f1_scores, dynamic_gflops, dynamic_active_percentages
):
    print(
        f"{'Dynamic U-Net':<22} {tau:>10.1f} {f1_score:>12.5f} "
        f"{gflops:>17.3f} {active_percentage:>13.2f}%"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
ax.scatter(
    [static_decoder_gflops],
    [static_f1],
    color="red",
    marker="*",
    s=180,
    label="Static U-Net",
)
ax.scatter(
    [attention_decoder_gflops],
    [attention_f1],
    color="blue",
    marker="s",
    s=80,
    label="Soft Attention U-Net",
)
ax.plot(
    dynamic_gflops,
    dynamic_f1_scores,
    color="green",
    marker="o",
    linestyle="-",
    label="Dynamic U-Net",
)
for tau, gflops, f1_score in zip(thresholds, dynamic_gflops, dynamic_f1_scores):
    ax.annotate(
        rf"$	au={tau:.1f}$",
        (gflops, f1_score),
        xytext=(5, 6),
        textcoords="offset points",
    )
ax.set_xlabel("Decoder GFLOPs")
ax.set_ylabel("F1 Score (Dice)")
ax.set_title("Decoder Compute vs. Road Segmentation Quality")
ax.grid(True, alpha=0.3)
ax.legend()

figure_directory = Path("results/figures")
figure_directory.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_directory / "pareto_frontier.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [ ]:
visual_inputs = None
visual_targets = None
for candidate_inputs, candidate_targets in test_loader:
    if bool(torch.any(candidate_targets > 0.5)):
        visual_inputs = candidate_inputs
        visual_targets = candidate_targets
        break
if visual_inputs is None:
    visual_inputs, visual_targets = next(iter(test_loader))

visual_inputs = visual_inputs.to(device)
visual_targets = visual_targets.to(device)
dynamic_model.set_entropy_threshold(0.5)
with torch.no_grad():
    static_logits = static_model(visual_inputs)
    dynamic_logits, auxiliary_logits_list, routing_masks_list = dynamic_model(
        visual_inputs, return_internals=True
    )
    _, entropy_map = dynamic_model.decoder1.entropy_gate(auxiliary_logits_list[-1])
    static_prediction = torch.sigmoid(static_logits) >= 0.5
    dynamic_prediction = torch.sigmoid(dynamic_logits) >= 0.5

mean_tensor = torch.tensor(TRAIN_MEAN, device=device).view(1, 3, 1, 1)
std_tensor = torch.tensor(TRAIN_STD, device=device).view(1, 3, 1, 1)
denormalized_image = (visual_inputs * std_tensor + mean_tensor).clamp(0.0, 1.0)
image_display = denormalized_image[0].permute(1, 2, 0).cpu().numpy()
target_display = visual_targets[0, 0].cpu().numpy()
static_display = static_prediction[0, 0].cpu().numpy()
dynamic_display = dynamic_prediction[0, 0].cpu().numpy()
entropy_display = entropy_map[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 5, figsize=(20, 4), constrained_layout=True)
axes[0].imshow(image_display)
axes[0].set_title("RGB Image")
axes[1].imshow(target_display, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Ground Truth")
axes[2].imshow(static_display, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Static U-Net")
axes[3].imshow(dynamic_display, cmap="gray", vmin=0, vmax=1)
axes[3].set_title("Dynamic U-Net")
axes[4].imshow(entropy_display, cmap="magma", vmin=0, vmax=1)
axes[4].set_title("Shannon Entropy")
for axis in axes:
    axis.axis("off")
fig.savefig(figure_directory / "segmentation_overlay.png", dpi=300, bbox_inches="tight")
plt.close(fig)